In [ ]:
!pip install langchain langchain-community langchain-openai langchain-text-splitters chromadb pypdf scikit-learn

In [ ]:
import os 
from dotenv import load_dotenv
load_dotenv(override=True)
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH="/Volumes/Work/Oracle_5_8/gpt-1paper.pdf"

# create the loader object
loader = PyPDFLoader(PDF_PATH)

docs = loader.load()

print(f"Total pages loaded: {len(docs)}")
print(f"Page content with first 500 characters: {docs[0].page_content[:500]}")


In [ ]:
# inspect pages

for i,doc in enumerate(docs):
    preview = doc.page_content[:100].replace("\n", " ")
    print(f"{i} | {len(doc.page_content)} | preview: {preview}")

In [ ]:
# text splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 80
)

chunks = text_splitter.split_documents(docs)
print(f"Chunks created: {len(chunks)}")
# sample chunk
print(f"Sample chunk content with first 500 characters: {chunks[0].page_content[:100]}")

In [ ]:
# inspect chunks

for i in range(4):
    print(f"Chunk : {i}")
    print(f"Metadata : {chunks[i].metadata}")
    print(f"Length : {len(chunks[i].page_content)}")
    print(f"Content : {chunks[i].page_content[:100]}")
    print("--------------------------------------------------")

In [ ]:
# embeddings 

from langchain_openai import OpenAIEmbeddings
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large")

#embed single chunk 
sample_chunk = chunks[0].page_content
sample_vector = embedding_model.embed_query(sample_chunk)
print(f"Sample vector text: {sample_chunk[:100]}")
print(f"Sample chunk vector length: {len(sample_vector)}")
print(f"First 10 dimensions of the sample vector: {sample_vector[:10]}")

In [ ]:
# store vectors in a vector database

from langchain_chroma import Chroma

print(f"Embedding {len(chunks)} chunks into vector database")

vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding = embedding_model,
    persist_directory = "./oracle_rag_db",
    collection_name = "gpt_1_paper"
)

total = vectorstore._collection.count()
print(f"Total Chunks stored in the database: {total}")

In [ ]:
# peek into the database

collection_data = vectorstore._collection.get(
    include = ["metadatas", "documents", "embeddings"],
)

ids = collection_data['ids']
embeddings = collection_data['embeddings']
metadatas = collection_data['metadatas']
documents = collection_data['documents']  

for i in range(5):
    print(f"Chunk ID: {ids[i]}")
    print(f"Metadata: {metadatas[i]}")
    print(f"Chunk content : {documents[i][:100]}")  # Print first 100 characters of the document
    print(f"Embedding length: {len(embeddings[i])}")
    print(f"First 10 dimensions of the embedding: {embeddings[i][:10]}")
    print("--------------------------------------------------")

In [ ]:
# dummy retireval function with cosine similarity
def retrieve_with_scores(query:str, k:int=3):
    "retrieve the top k chinks with cosine scores"
    results= vectorstore.similarity_search_with_relevance_scores(query,k=k)
    print(f"Query: {query}")
    print(f"Top {k} results: \n")
    for i,(doc,score) in enumerate(results):
        print(f"Rank {i+1} | score: {score : .4f} | Page : {doc.metadata.get('page', '?')}")
        print(f"{doc.page_content[:200]}")  # Print first 200 characters of the document
        print()
    return results

In [ ]:
result = retrieve_with_scores("Which datasets were used for Natural language inference? ")

print(result)

In [ ]:
# RAG

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOpenAI(model="gpt-4o",temperature=0)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

def rag_answer(query:str, show_sources:bool=True) -> str:
    # step 1: retrieve relevant chunks
    retrieved = retriever.invoke(query)
    # step 2: build context 
    context = "\n\n".join(
        f"[Page {doc.metadata.get('page', '?')}]\n{doc.page_content}" 
        for doc in retrieved
    )
    # step 3 : build prompt
    messages = [
        SystemMessage(content=(
            "You are a precise research paper assistant. "
            "Answer the question using ONLY the provided context from the GPT-1 paper. "
            "If the answer is not in the context, say: 'This is not covered in the retrieved sections.' "
            "Be specific, cite page numbers where possible, and do not hallucinate."
        )),
        HumanMessage(content=f"Context from GPT-1 paper:\n{context}\n\nQuestion: {query}")
    ]
    
    # strep 4 :call the LLM
    response = llm.invoke(messages)
    
    if show_sources:
        print("Sources used for the answer:\n")
        for i,doc in enumerate(retrieved):
            print(f"Source {i+1} | Page: {doc.metadata.get('page', '?')}")
            print(f"{doc.page_content[:200]}")  # Print first 200 characters of the document
            print("--------------------------------------------------")
    return response.content

In [ ]:
# test 

q1 = "What two stage training procedure was used for GPT-1?"
ans1 = rag_answer(q1)
print(f"Answer to '{q1}':\n{ans1}\n")

In [ ]:
q2 = "How does GPT-1 handle different tasks during fine tuning?"
ans2 = rag_answer(q2)
print(f"Answer to '{q2}':\n{ans2}\n")